In [0]:
#The KPI snapshot table
#Build the snapshot
from pyspark.sql import functions as F

CATALOG = "capstone_supply_chain"
C = CATALOG

df_kpi = spark.sql(f"""
    SELECT
        current_timestamp() AS snapshot_timestamp,

        -- Procurement
        (SELECT COUNT(*) FROM {C}.gold.fact_purchase_orders)                    AS total_purchase_orders,
        (SELECT COUNT(DISTINCT vendor_id) FROM {C}.gold.fact_purchase_orders)   AS active_vendors,
        (SELECT COUNT(DISTINCT currency) FROM {C}.gold.fact_purchase_orders)    AS currencies_in_use,
        (SELECT ROUND(AVG(lead_time_days), 1) FROM {C}.gold.fact_purchase_orders) AS avg_lead_time_days,

        -- Vendors
        (SELECT ROUND(AVG(composite_score), 1) FROM {C}.gold.agg_vendor_scorecard) AS avg_vendor_score,
        (SELECT COUNT(*) FROM {C}.gold.agg_vendor_scorecard WHERE risk_rating = 'High') AS high_risk_vendors,

        -- Inventory
        (SELECT ROUND(SUM(total_inventory_value), 2) FROM {C}.gold.agg_inventory_health) AS total_inventory_value_usd,
        (SELECT SUM(critical_items) FROM {C}.gold.agg_inventory_health)         AS critical_stock_items,
        (SELECT SUM(reorder_items) FROM {C}.gold.agg_inventory_health)          AS reorder_needed_items,

        -- Deliveries (on time = arrived on or before the scheduled date)
        (SELECT COUNT(*) FROM {C}.silver.deliveries WHERE actual_date_clean IS NOT NULL) AS deliveries_completed,
        (SELECT COUNT(*) FROM {C}.silver.deliveries WHERE actual_date_clean IS NULL)     AS deliveries_pending,
        (SELECT ROUND(100.0 * SUM(CASE WHEN days_early_late <= 0 THEN 1 ELSE 0 END) / COUNT(*), 1)
           FROM {C}.silver.deliveries WHERE actual_date_clean IS NOT NULL)      AS on_time_delivery_pct,

        -- Quality
        (SELECT COUNT(*) FROM {C}.silver.quality_inspections WHERE is_open)     AS open_quality_issues,
        (SELECT COUNT(*) FROM {C}.silver.quality_inspections WHERE is_open AND severity_rank = 3) AS open_critical_issues
""")

df_kpi.write.mode("overwrite").saveAsTable(f"{C}.gold_consolidated.supply_chain_kpi_snapshot")
print("✅ gold_consolidated.supply_chain_kpi_snapshot created")
df_kpi.show(truncate=False, vertical=True)

In [0]:
#Check every KPI against its source
C = "capstone_supply_chain"

kpi = spark.table(f"{C}.gold_consolidated.supply_chain_kpi_snapshot").first()

po  = spark.table(f"{C}.silver.purchase_orders")
inv = spark.table(f"{C}.silver.inventory")
dl  = spark.table(f"{C}.silver.deliveries")
qi  = spark.table(f"{C}.silver.quality_inspections")
sc  = spark.table(f"{C}.gold.agg_vendor_scorecard")

done = dl.where(F.col("actual_date_clean").isNotNull())

checks = {
    "total_purchase_orders":     (kpi.total_purchase_orders, po.count()),
    "active_vendors":            (kpi.active_vendors, po.select("vendor_id").distinct().count()),
    "currencies_in_use":         (kpi.currencies_in_use, po.select("currency").distinct().count()),
    "avg_lead_time_days":        (kpi.avg_lead_time_days, round(po.agg(F.avg("lead_time_days")).first()[0], 1)),
    "avg_vendor_score":          (kpi.avg_vendor_score, round(sc.agg(F.avg("composite_score")).first()[0], 1)),
    "high_risk_vendors":         (kpi.high_risk_vendors, sc.where(F.col("risk_rating") == "High").count()),
    "total_inventory_value_usd": (kpi.total_inventory_value_usd, round(inv.agg(F.sum("inventory_value_usd")).first()[0], 2)),
    "critical_stock_items":      (kpi.critical_stock_items, inv.where(F.col("stock_status") == "CRITICAL").count()),
    "reorder_needed_items":      (kpi.reorder_needed_items, inv.where(F.col("stock_status") == "REORDER").count()),
    "deliveries_completed":      (kpi.deliveries_completed, done.count()),
    "deliveries_pending":        (kpi.deliveries_pending, dl.count() - done.count()),
    "on_time_delivery_pct":      (kpi.on_time_delivery_pct,
                                  round(done.where(F.col("delivery_status").isin("On Time", "Early")).count() / done.count() * 100, 1)),
    "open_quality_issues":       (kpi.open_quality_issues, qi.where(F.col("status_clean") == "Open").count()),
    "open_critical_issues":      (kpi.open_critical_issues,
                                  qi.where((F.col("status_clean") == "Open") & (F.col("severity_clean") == "Critical")).count()),
}

for name, (snapshot_value, recomputed) in checks.items():
    ok = abs(float(snapshot_value) - float(recomputed)) < 0.01
    print(("✅" if ok else "❌"), f"{name}: snapshot={snapshot_value}  recomputed={recomputed}")

# Gold layer inventory
print("\nGOLD LAYER TABLES")
for schema in ["gold", "gold_consolidated"]:
    for t in spark.sql(f"SHOW TABLES IN {C}.{schema}").collect():
        print(f"  {schema}.{t.tableName}: {spark.table(f'{C}.{schema}.{t.tableName}').count()} rows")